In [ ]:
import os
import bs4
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['HUGGINGFACE_API_KEY'] = os.getenv("HUGGINGFACE_API_KEY")

# Embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load Wikipedia page
loader = WebBaseLoader(
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    # bs_kwargs=dict(
    #     parse_only=bs4.SoupStrainer(class_=('mw-content-text'))
    # )
)
documents = loader.load()
print(f" Loaded {len(documents)} documents")

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_documents = splitter.split_documents(documents)
print(f" Split into {len(splitted_documents)} chunks")

# Vectorstore + Retriever
# vectorstore = Chroma.from_documents(splitted_documents, hf_embeddings)

vectorstore = FAISS.from_documents(splitted_documents, hf_embeddings)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile")

# Prompt — use {question} not {input} for LCEL chains
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based on the context below:\n\n{context}"),
    ("user", "{question}")
])

#  format_docs — converts retrieved Document objects to plain text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#  
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),  # retrieve + format docs
        "question": RunnablePassthrough()                     # pass question as-is
    } | prompt | llm | StrOutputParser()
)

# Test
response = rag_chain.invoke("What is Artificial Intelligence?")
print(response)

 Loaded 1 documents
 Split into 309 chunks
According to the text, Artificial Intelligence (AI) refers to the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

In simpler terms, AI can be defined as:

* "The computational part of the ability to achieve goals in the world" (McCarthy)
* "The ability to solve hard problems" (Marvin Minsky)
* The study of agents that perceive their environment and take actions that maximize their chances of achieving defined goals (Artificial Intelligence: A Modern Approach)
